# ImageJ-Style Preprocessing (Python)

This notebook reproduces the classic ImageJ preprocessing workflow used for
fluorescence nanoparticle movies — **without Java/pyimagej**. Each step maps to a
familiar ImageJ command:

| Step | ImageJ command | Python equivalent here |
|---|---|---|
| Background removal | *Process → Subtract Background* (rolling ball) | `skimage.morphology.opening` with a ball footprint, then subtract |
| Denoise | *Process → Filters → Median* | `skimage.filters.median` |
| Threshold | *Image → Adjust → Auto Threshold* (Otsu) | `skimage.filters.threshold_otsu` |

The goal is not byte-for-byte ImageJ reproduction, but an equivalent Python
preprocessing path that feeds `nanotrack`'s detection stage.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import filters, morphology

plt.rcParams['figure.dpi'] = 90


In [ ]:
def rolling_ball_background(image, radius=10):
    """Estimate the background by opening the image with a ball (rolling ball)."""
    ball = morphology.ball(radius)
    return morphology.opening(image, footprint=ball)


def imagej_style_preprocess(frame):
    """rolling ball background -> median denoise -> Otsu threshold."""
    background = rolling_ball_background(frame, radius=10)
    flat = frame - background
    median = filters.median(flat, footprint=morphology.disk(2))
    thresh = filters.threshold_otsu(median)
    return np.clip(flat, 0, 255).astype(np.uint8), median, median > thresh


In [ ]:
# Load a real 8-bit fluorescence movie from the local reference archive
# (fall back to synthetic data if the archive is not present).
import sys, tempfile, zipfile
from pathlib import Path
from nanotrack.synth import generate
from nanotrack.config import PipelineConfig
from nanotrack.io import load_video

repo = Path.cwd()
archive = repo / 'ref' / 'SWNTs trackingV3.zip'
if archive.exists():
    with zipfile.ZipFile(archive) as z:
        z.extract('SWNTs trackingV3/sample data/Artificial8bit/Artificial8bit.tif', f'{tempfile.gettempdir()}/nt_art8')
    frames, _ = load_video(f'{tempfile.gettempdir()}/nt_art8/SWNTs trackingV3/sample data/Artificial8bit/Artificial8bit.tif')
    print('using real Artificial8bit sample:', frames.shape)
else:
    frames, _ = generate(PipelineConfig(image_size=256, n_frames=10, seed=0))
    print('archive not found; using synthetic sample:', frames.shape)


In [ ]:
frame = frames[0]
flat, median, mask = imagej_style_preprocess(frame)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
titles = ['raw', 'after rolling-ball', 'after median', 'Otsu mask']
for ax, img, t in zip(axes, [frame, flat, median, mask.astype(np.uint8) * 255], titles):
    ax.imshow(img, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()
print(f'foreground fraction: {mask.mean():.4f}')


## Notes
- `morphology.opening` with a ball footprint is the morphological equivalent of
  ImageJ's rolling-ball background subtraction; subtract it from the raw frame.
- Median filtering removes salt-and-pepper noise before thresholding.
- Otsu's method picks the threshold automatically (ImageJ *Auto Threshold* default).
- This is the *visualization* path; the library's `preprocess(..., backend='skimage')`
  uses the same building blocks for the production pipeline.
